# QF-Attack — Tyro test bench (FIT5230, Theme 2 Dark)

**Team Tyro** · Echo Zhao, Nissa Colidea · reference paper: *A Pilot Study of Query-Free
Adversarial Attack against Stable Diffusion* (Zhuang et al., CVPRW 2023, [arXiv:2303.16378](https://arxiv.org/abs/2303.16378))

---

## Read this before you run anything

**What QF-Attack is.** You append about **five junk characters** to a text prompt.
The image Stable Diffusion produces changes dramatically — a concept vanishes, or the
whole scene drifts. No access to the diffusion model is needed; the weakness is in the
**CLIP text encoder** that reads the prompt.

> **Metaphor.** Stable Diffusion is a chef who cannot hear well. The text encoder is the
> waiter who relays your order. QF-Attack doesn't bribe the chef — it mumbles five
> syllables at the waiter, and a completely different dish arrives.

**What QF-Attack is NOT — and why this matters for our project.**
Our project attacks **PhotoGuard**, a shield painted onto *someone's photograph*. QF never
touches an image. There is no protected asset, no defender, and no shield to strip. So:

| | axis | protected asset | who is harmed |
|---|---|---|---|
| **PhotoGuard** (our target) | image | a person's photo | — (it is the shield) |
| **IMPRESS** (our baseline) | image | — | the shield |
| **QF-Attack** (this notebook) | **text** | **none** | the person typing the prompt |

QF is therefore **not** a candidate baseline for our Milestone 1/2/3 pipeline. We run it
here for two specific, mark-bearing reasons:

1. **M4, "Unconstrained System Enhancement & Feasibility" (4%)** — our stated
   no-limits idea is a *two-flank attack*: strip the image-side shield **and** perturb the
   text side. PhotoGuard guards the picture; **nobody is guarding the prompt.** This
   notebook is the feasibility demo that rubric asks for.
2. **M4, "Adversarial Role-Reversal Strategy & Demo" (5%)** — having actually run a
   text-side attack lets us describe a text-side *defence* concretely instead of vaguely.

**Time-box this.** One session. Save the figures, write three lines in the strategy log,
go back to the IMPRESS pipeline. Do not let this grow.

## What this notebook does differently from the paper

We implement the attack from scratch on top of `diffusers`/`transformers` rather than
cloning the 2023 repo (the IMPRESS clone cost us an afternoon of dependency archaeology —
see `HANDOVER.md`). Our version adds:

- a **composite targeted objective** — the paper pushes away from the concept being erased;
  we simultaneously *hold on to* the rest of the prompt (`lambda_keep`), so the attack
  deletes one object instead of wrecking the whole image;
- a **random-suffix control arm**, so we can prove the effect comes from the *optimisation*
  and not merely from adding gibberish;
- **independent evaluation** — the attack optimises against SD's own CLIP ViT-L/14 text
  encoder, but we score results with a *different* CLIP (ViT-B/32) to avoid marking our
  own homework.


## 0 · Runtime check

Runtime ▸ Change runtime type ▸ **T4 GPU**. This notebook fits comfortably in free-tier Colab: the search is text-encoder only (seconds), and only the final image generation touches the U-Net.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on the T4: Runtime > Change runtime type > T4 GPU"


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors matplotlib
print("done")


## 1 · Load the text encoder

We load **only** the tokenizer and text encoder first. This is the part QF attacks, and
it is small — the whole search below runs without the image model in memory.

`stable-diffusion-v1-5/stable-diffusion-v1-5` is the community mirror. The original
`runwayml/...` repo was deleted from Hugging Face in 2024, which is what broke the IMPRESS
scripts; we use the mirror everywhere for consistency. No HF login required.

In [ ]:
import torch, string, random, numpy as np
from transformers import CLIPTokenizer, CLIPTextModel

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DEVICE   = "cuda"

tokenizer    = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float32
).to(DEVICE).eval()

for p in text_encoder.parameters():
    p.requires_grad_(False)

print("text encoder loaded:", sum(p.numel() for p in text_encoder.parameters())/1e6, "M params")


## 2 · The attack, in code

Two helpers and one search loop. That is the entire method.

**`embed(prompts)`** turns text into the 77x768 tensor that Stable Diffusion's
cross-attention actually consumes, flattened so we can take a cosine similarity over the
whole thing. We use `last_hidden_state`, *not* the pooled vector, because the pooled
vector is not what conditions the diffusion — attacking it would be attacking the wrong
quantity.

**`greedy_search`** is coordinate descent over five characters. For each position, try
every character in the charset, keep whichever moves the embedding furthest in the
direction we want, then move to the next position. Two passes are usually enough.

> **Metaphor.** Five dials on a safe. You cannot solve them jointly, so you spin dial 1
> through every setting listening for the click, lock it in, move to dial 2, and repeat.
> Crude — but the safe is badly made, and it opens.

In [ ]:
CHARSET = list(string.ascii_lowercase + string.digits)

@torch.no_grad()
def embed(prompts, batch=64):
    """Text -> flattened (N, 77*768) conditioning tensor, exactly what SD consumes."""
    if isinstance(prompts, str):
        prompts = [prompts]
    outs = []
    for i in range(0, len(prompts), batch):
        toks = tokenizer(
            prompts[i:i+batch], padding="max_length",
            max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
        ).to(DEVICE)
        outs.append(text_encoder(**toks).last_hidden_state.flatten(1))
    return torch.cat(outs)

def cos(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=-1)


In [ ]:
def greedy_search(prompt, k=5, rounds=2, target=None, keep=None,
                  lambda_keep=1.0, charset=CHARSET, verbose=True):
    """Find a k-character suffix that maximally disturbs the prompt embedding.

    untargeted (target=None) : push the embedding AWAY from the clean prompt.
    targeted   (target=...)  : push AWAY from the concept to erase, while
                               (if keep is given) HOLDING ON to the rest of the
                               prompt -- this composite objective is our addition;
                               the paper optimises the first term only.
    """
    e_ref  = embed(prompt)
    e_tgt  = embed(target) if target is not None else None
    e_keep = embed(keep)   if keep   is not None else None

    suffix = ["a"] * k
    history = []

    def score(cands):
        e = embed(cands)
        if e_tgt is None:
            s = -cos(e, e_ref)                      # untargeted: maximise drift
        else:
            s = -cos(e, e_tgt)                      # erase the target concept
            if e_keep is not None:
                s = s + lambda_keep * cos(e, e_keep)  # ...but keep everything else
        return s

    for r in range(rounds):
        for pos in range(k):
            cands = []
            for c in charset:
                trial = suffix.copy(); trial[pos] = c
                cands.append(prompt + " " + "".join(trial))
            s = score(cands)
            best = int(s.argmax())
            suffix[pos] = charset[best]
            history.append(float(s[best]))
        if verbose:
            print(f"  round {r+1}/{rounds}: suffix={''.join(suffix)!r}  score={history[-1]:.4f}")

    adv = prompt + " " + "".join(suffix)
    drift = float(1 - cos(embed(adv), e_ref))
    return {"suffix": "".join(suffix), "adv_prompt": adv,
            "embedding_drift": drift, "history": history}

def random_suffix(prompt, k=5, seed=0):
    """CONTROL ARM: same length of gibberish, no optimisation."""
    rng = random.Random(seed)
    s = "".join(rng.choice(CHARSET) for _ in range(k))
    adv = prompt + " " + s
    return {"suffix": s, "adv_prompt": adv,
            "embedding_drift": float(1 - cos(embed(adv), embed(prompt)))}


## 3 · Untargeted attack — five characters, measured

Pick a prompt and run the search. Watch `embedding_drift`: 0 means "identical meaning to
the encoder", larger means "the encoder now reads something else".

The control arm is the important scientific bit. If random gibberish produced the same
drift, there would be no attack here — just noise. It doesn't, and the gap is the finding.

In [ ]:
PROMPT = "a photograph of a dog sitting on a wooden bench in a park"

print("optimising...")
atk  = greedy_search(PROMPT, k=5, rounds=2)
ctrl = random_suffix(PROMPT, k=5, seed=1)

print()
print(f"clean prompt      : {PROMPT!r}")
print(f"QF adversarial    : {atk['adv_prompt']!r}   drift = {atk['embedding_drift']:.4f}")
print(f"random control    : {ctrl['adv_prompt']!r}   drift = {ctrl['embedding_drift']:.4f}")
print()
print(f"--> optimisation buys {atk['embedding_drift']/max(ctrl['embedding_drift'],1e-9):.1f}x "
      f"more embedding drift than gibberish of the same length.")


## 4 · Load Stable Diffusion and generate

Now we pay the GPU cost. **Same seed for every image** — the only variable across the
three panels is the prompt text, so any difference you see is caused by the five
characters and nothing else. This is the single most important control in the notebook;
without a fixed seed the comparison proves nothing.

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

SEED  = 5230
STEPS = 25

def generate(prompt, seed=SEED):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    return pipe(prompt, num_inference_steps=STEPS, guidance_scale=7.5, generator=g).images[0]

print("pipeline ready")


In [ ]:
import matplotlib.pyplot as plt

panels = [
    ("clean prompt",        PROMPT),
    ("QF attack (5 chars)", atk["adv_prompt"]),
    ("random control",      ctrl["adv_prompt"]),
]

images = []
for label, p in panels:
    print("generating:", label)
    images.append(generate(p))

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (label, p), im in zip(axes, panels, images):
    ax.imshow(im); ax.axis("off")
    ax.set_title(label, fontsize=11, pad=8)
    ax.text(0.5, -0.06, f"...{p[-28:]!r}", transform=ax.transAxes,
            ha="center", va="top", fontsize=8, color="#555555", family="monospace")
fig.suptitle("QF-Attack: identical seed, identical model, five characters of difference",
             fontsize=12)
fig.tight_layout()
fig.savefig("qf_untargeted.png", dpi=150, bbox_inches="tight")
plt.show()


## 5 · Score it with an *independent* CLIP

The attack was optimised against SD's own text encoder (CLIP ViT-L/14). Scoring the result
with that same encoder would be marking our own homework — of course the thing we optimised
looks optimised. We load a **different** CLIP (ViT-B/32) and ask it a neutral question:
*how well does each generated image still match the original prompt?*

Lower score = the image drifted further from what the user asked for = stronger attack.

In [ ]:
# FIXED
from transformers import CLIPModel, CLIPProcessor

EVAL_ID = "openai/clip-vit-base-patch32"     # deliberately NOT the encoder we attacked
clip_model = CLIPModel.from_pretrained(EVAL_ID).to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained(EVAL_ID)

@torch.no_grad()
def clip_score(image, text):
    """Cosine similarity between an image and a text, in independent CLIP space."""
    inp = clip_proc(text=[text], images=[image], return_tensors="pt",
                    padding=True, truncation=True).to(DEVICE)

    # Get image and text features. The error indicates these return BaseModelOutputWithPooling.
    # We need to extract the actual tensor from their .pooler_output attribute.
    image_features_output = clip_model.get_image_features(pixel_values=inp["pixel_values"])
    text_features_output = clip_model.get_text_features(
        input_ids=inp["input_ids"],
        attention_mask=inp["attention_mask"] # Corrected typo: attention_ids -> attention_mask
    )

    # Extract the actual tensor features from the output objects
    ie = image_features_output.pooler_output if hasattr(image_features_output, 'pooler_output') else image_features_output
    te = text_features_output.pooler_output if hasattr(text_features_output, 'pooler_output') else text_features_output

    ie = ie / ie.norm(dim=-1, keepdim=True)
    te = te / te.norm(dim=-1, keepdim=True)
    return float((ie @ te.T).squeeze())

scores = [clip_score(im, PROMPT) for im in images]
for (label, _), s in zip(panels, scores):
    print(f"{label:22s}  CLIP(image, ORIGINAL prompt) = {s:.4f}")

base = scores[0]
print()
print(f"QF attack     dropped alignment by {base - scores[1]:+.4f} ({100*(scores[1]-base)/base:+.1f}%)")
print(f"random control dropped it by       {base - scores[2]:+.4f} ({100*(scores[2]-base)/base:+.1f}%)")

In [ ]:
# from transformers import CLIPModel, CLIPProcessor

# EVAL_ID = "openai/clip-vit-base-patch32"     # deliberately NOT the encoder we attacked
# clip_model = CLIPModel.from_pretrained(EVAL_ID).to(DEVICE).eval()
# clip_proc  = CLIPProcessor.from_pretrained(EVAL_ID)

# @torch.no_grad()
# def clip_score(image, text):
#     """Cosine similarity between an image and a text, in independent CLIP space."""
#     inp = clip_proc(text=[text], images=[image], return_tensors="pt",
#                     padding=True, truncation=True).to(DEVICE)
#     ie = clip_model.get_image_features(pixel_values=inp["pixel_values"])
#     te = clip_model.get_text_features(input_ids=inp["input_ids"],
#                                       attention_mask=inp["attention_mask"])
#     ie = ie / ie.norm(dim=-1, keepdim=True)
#     te = te / te.norm(dim=-1, keepdim=True)
#     return float((ie @ te.T).squeeze())

# scores = [clip_score(im, PROMPT) for im in images]
# for (label, _), s in zip(panels, scores):
#     print(f"{label:22s}  CLIP(image, ORIGINAL prompt) = {s:.4f}")

# base = scores[0]
# print()
# print(f"QF attack     dropped alignment by {base - scores[1]:+.4f} ({100*(scores[1]-base)/base:+.1f}%)")
# print(f"random control dropped it by       {base - scores[2]:+.4f} ({100*(scores[2]-base)/base:+.1f}%)")


In [ ]:
# Palette: Okabe-Ito subset, validated colourblind-safe (deutan dE 11.0, normal dE 25.8).
PAL = ["#0072B2", "#D55E00", "#009E73"]

fig, ax = plt.subplots(figsize=(6.5, 4))
labels = [p[0] for p in panels]
bars = ax.bar(labels, scores, color=PAL, width=0.55)
for b, s in zip(bars, scores):                     # direct labels, not an extra axis
    ax.text(b.get_x() + b.get_width()/2, s + 0.004, f"{s:.3f}",
            ha="center", fontsize=10, color="#333333")
ax.set_ylabel("CLIP similarity to the ORIGINAL prompt")
ax.set_title("Lower = the five characters pushed the image further off-brief")
ax.set_ylim(0, max(scores) * 1.18)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e8e8e8", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("qf_clipscore.png", dpi=150, bbox_inches="tight")
plt.show()


## 6 · Targeted attack — delete one object, keep the rest

The untargeted attack is a wrecking ball. The targeted attack is a scalpel: erase *the dog*
and leave *the bench and the park* intact.

This is where our composite objective earns its keep. Optimising only "move away from
`dog`" also moves away from bench, park, and photograph — you get mush. Adding
`lambda_keep * cos(e, e_keep)` anchors the rest of the sentence in place.

> **Metaphor.** Untargeted is throwing out the whole grocery bag because it contains one
> bad apple. Targeted-with-keep is reaching in and removing only the apple.

In [ ]:
PROMPT_T = "a photograph of a dog and a cat sitting on a wooden bench"
ERASE    = "a dog"                                   # concept to remove
KEEP     = "a photograph of a cat sitting on a wooden bench"   # what must survive

print("optimising targeted attack...")
atk_t = greedy_search(PROMPT_T, k=5, rounds=2, target=ERASE, keep=KEEP, lambda_keep=1.0)
print()
print("adversarial prompt:", repr(atk_t["adv_prompt"]))

img_clean_t = generate(PROMPT_T)
img_adv_t   = generate(atk_t["adv_prompt"])

fig, axes = plt.subplots(1, 2, figsize=(9, 4.8))
for ax, im, t in zip(axes, [img_clean_t, img_adv_t],
                     ["clean prompt", f"targeted QF (erase {ERASE!r})"]):
    ax.imshow(im); ax.axis("off"); ax.set_title(t, fontsize=11)
fig.tight_layout()
fig.savefig("qf_targeted.png", dpi=150, bbox_inches="tight")
plt.show()

print()
print("Did we erase the target while keeping the rest?  (independent CLIP)")
for name, im in [("clean", img_clean_t), ("attacked", img_adv_t)]:
    print(f"  {name:9s} vs ERASED concept {ERASE!r:22s}: {clip_score(im, ERASE):.4f}")
    print(f"  {name:9s} vs KEPT  concept 'a cat'               : {clip_score(im, 'a cat'):.4f}")
print()
print("A successful targeted attack: the ERASED row drops sharply, the KEPT row barely moves.")


## 7 · Findings — write these down before you close the tab

Fill this in from *your* run, then paste the three lines into `Echo-M4-Strategy-Log.md`.

| observation | value from this run |
|---|---|
| embedding drift, optimised vs random control | ______ vs ______ |
| CLIP alignment drop, untargeted attack | ______ |
| targeted: erased-concept score before → after | ______ → ______ |
| targeted: kept-concept score before → after | ______ → ______ |
| wall-clock for the search (text encoder only) | ______ |

### What this buys us (the part that carries marks)

**Cost asymmetry is the headline.** The QF search touches only the text encoder — seconds
on a T4. PhotoGuard's diffusion attack costs us roughly **77 minutes per image**. Same
family of goal (disrupt what the model produces), four orders of magnitude apart in price.
That contrast belongs in the M4 report.

**The undefended flank.** PhotoGuard, Glaze and Mist all guard the *pixels*. None of them
guard the *prompt*. A two-flank attack — purify the image to strip the shield, then perturb
the prompt to steer the edit — is our M4 "unconstrained enhancement", and this notebook is
its feasibility demo.

**Role reversal (M4, 5%).** Having run the attack, the defence writes itself and we can
say it concretely: normalise prompts before encoding (strip non-lexical character runs),
reject prompts whose embedding sits far from any in-vocabulary neighbourhood, or ensemble
two text encoders and refuse when they disagree. We can prototype the first one in ten
lines — that is the "functional demo" the rubric asks for.

### What this does NOT buy us

It is **not** a Milestone 1–3 baseline. Wrong axis (text, not image), no protected asset,
and its metrics do not share an evaluation harness with `pg_metric.py`. It stays in Related
Work and in the M4 report. Close the tab and go back to the IMPRESS pipeline.
